# Explore Parquets
Load existing data, no reprocessing needed.

In [1]:
import pandas as pd
import numpy as np

BASE = '/home/nikita/code/PlatoIsDead/notebook_prototype/sentiment_analysis'

booking_risk   = pd.read_parquet(f'{BASE}/booking_risk.parquet')
guest_enriched = pd.read_parquet(f'{BASE}/guest_enriched.parquet')
hotel_summary  = pd.read_parquet(f'{BASE}/hotel_summary.parquet')

print('booking_risk:  ', booking_risk.shape)
print('guest_enriched:', guest_enriched.shape)
print('hotel_summary: ', hotel_summary.shape)

booking_risk:   (7982, 22)
guest_enriched: (56415, 13)
hotel_summary:  (5, 10)


In [2]:
# ── Hotel summary ─────────────────────────────────────────────────────────────
print('=== Hotel Summary ===')
hotel_summary

=== Hotel Summary ===


,hotel_id,n_bookings,avg_risk,high_risk_bookings,median_reply_min,avg_neg_share,threats,complaints,problems,high_risk_share_%
0,1,2488,25.6,273,18.80,19.5,2,35,492,11.0
1,2,1009,21.7,118,4.40,19.5,1,8,248,11.7
2,3,813,31.7,162,6.45,29.2,0,0,12,19.9
3,4,227,20.5,19,21.40,14.3,0,0,35,8.4
4,5,3445,22.9,246,4.40,17.2,9,110,1131,7.1


In [3]:
# ── Risk distribution per hotel ───────────────────────────────────────────────
risk = (
    booking_risk
    .groupby(['hotel_id', 'risk_level'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
for col in ['LOW','MEDIUM','HIGH']:
    if col not in risk.columns: risk[col] = 0
risk['total'] = risk[['LOW','MEDIUM','HIGH']].sum(axis=1)
risk['HIGH_%']   = (risk['HIGH']   / risk['total'] * 100).round(1)
risk['MEDIUM_%'] = (risk['MEDIUM'] / risk['total'] * 100).round(1)
print('=== Risk Distribution per Hotel ===')
risk

=== Risk Distribution per Hotel ===


/tmp/ipykernel_33410/910616524.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['hotel_id', 'risk_level'])


risk_level,hotel_id,LOW,MEDIUM,HIGH,total,HIGH_%,MEDIUM_%
0,1,1649,566,273,2488,11.0,22.7
1,2,725,166,118,1009,11.7,16.5
2,3,471,180,162,813,19.9,22.1
3,4,175,33,19,227,8.4,14.5
4,5,2342,857,246,3445,7.1,24.9


In [4]:
# ── Top topics per hotel ──────────────────────────────────────────────────────
topics = (
    booking_risk
    .groupby(['hotel_id', 'top_topic'])
    .size()
    .reset_index(name='n_bookings')
    .sort_values(['hotel_id', 'n_bookings'], ascending=[True, False])
)
print('=== Top Topics per Hotel ===')
print(topics.groupby('hotel_id').head(5).to_string(index=False))

=== Top Topics per Hotel ===
 hotel_id   top_topic  n_bookings
        1      прочее        1159
        1     вопросы         450
        1     чистота         253
        1 заезд-выезд         196
        1     платежи         110
        2      прочее         428
        2     вопросы         225
        2 заезд-выезд         132
        2      ремонт          51
        2 температура          48
        3      прочее         739
        3    интернет          42
        3      ремонт           9
        3     вопросы           8
        3     чистота           8
        4      прочее          85
        4     вопросы          41
        4 заезд-выезд          33
        4     чистота          22
        4    интернет          15
        5      прочее        2152
        5     вопросы         475
        5 заезд-выезд         229
        5     чистота         205
        5    интернет          97


In [5]:
# ── Reply time distribution ───────────────────────────────────────────────────
reply = (
    booking_risk
    .groupby(['hotel_id', 'reply_bucket'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
print('=== Reply Time Buckets per Hotel ===')
reply

=== Reply Time Buckets per Hotel ===


/tmp/ipykernel_33410/2653155779.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['hotel_id', 'reply_bucket'])


reply_bucket,hotel_id,<=5m,5-15m,15-60m,>60m
0,1,563,364,518,592
1,2,424,161,121,103
2,3,273,36,78,179
3,4,54,14,50,39
4,5,1658,607,528,333


In [6]:
# ── Message type distribution in guest_enriched ───────────────────────────────
print('=== Message Types (guest_enriched) ===')
print(guest_enriched['msg_type'].value_counts())
print()
print('=== Topics (guest_enriched) ===')
print(guest_enriched['topic'].value_counts())

=== Message Types (guest_enriched) ===
msg_type
OTHER        54332
PROBLEM       1918
COMPLAINT      153
THREAT          12
Name: count, dtype: int64

=== Topics (guest_enriched) ===
topic
прочее         32427
вопросы         7062
чистота         4967
заезд-выезд     3418
платежи         2513
парковка        1259
ремонт          1127
интернет        1079
шум              909
удобства         642
температура      628
правила          295
персонал          89
Name: count, dtype: int64


In [7]:
# ── Sentiment over time (monthly) ─────────────────────────────────────────────
ge = guest_enriched.copy()
ge['month'] = ge['date_add'].dt.to_period('M').astype(str)

monthly = (
    ge.groupby(['hotel_id', 'month', 'hf_sentiment'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
for col in ['NEG','NEU','POS']:
    if col not in monthly.columns: monthly[col] = 0
monthly['total'] = monthly[['NEG','NEU','POS']].sum(axis=1)
monthly['NEG_%'] = (monthly['NEG'] / monthly['total'] * 100).round(1)
print(monthly[['hotel_id','month','total','NEG_%']].tail(20).to_string(index=False))

 hotel_id   month  total  NEG_%
        5 2024-02    657   21.0
        5 2024-03   1207   16.0
        5 2024-04   1221   19.2
        5 2024-05    787   17.4
        5 2024-06    856   17.6
        5 2024-07    890   22.9
        5 2024-08    732   20.5
        5 2024-09    656   17.2
        5 2024-10    830   19.4
        5 2024-11   1000   15.7
        5 2024-12   1037   17.7
        5 2025-01   1073   16.9
        5 2025-02   2127   18.9
        5 2025-03   2958   21.4
        5 2025-04   2791   19.2
        5 2025-05   3768   20.0
        5 2025-06   4080   18.6
        5 2025-07   2933   15.5
        5 2025-08   3097   17.6
        5 2025-09    133   16.5


In [8]:
# ── HIGH risk bookings — what are they about? ─────────────────────────────────
high = booking_risk[booking_risk['risk_level'] == 'HIGH'].copy()
print(f'HIGH risk bookings: {len(high)}')
print()
print('Topics:')
print(high['top_topic'].value_counts().head(10))
print()
print('Hotels:')
print(high['hotel_id'].value_counts())

HIGH risk bookings: 818

Topics:
top_topic
прочее         405
заезд-выезд     80
ремонт          77
чистота         60
вопросы         53
платежи         43
шум             27
парковка        23
температура     18
интернет        12
Name: count, dtype: int64

Hotels:
hotel_id
1    273
5    246
3    162
2    118
4     19
Name: count, dtype: int64
